# 02g. Aplicar Splink 5 no Spark — mãe + endereço (MA)

Mesma lógica do [`02f_aplicar_splink_spark_luis.ipynb`](02f_aplicar_splink_spark_luis.ipynb),
com os parquets de aplicação do Maranhão do
[`02d_aplicar_splink_mae_endereco.ipynb`](02d_aplicar_splink_mae_endereco.ipynb).

**Linker / predict / export no Spark** (`SparkAPI`) — não DuckDB local.

Paths em `/data/spark/shared/datasets/` (copiar do `OUTPUT_DIR/uf_21`):
- `censo_limpo_aplicacao.parquet` / `cpf_limpo_aplicacao.parquet`
- `splink_model_mae_endereco.json`
- saída: `splink_predictions_mae_endereco_spark.parquet`

Mesmas 14 regras de blocking do 02d; `predict` com
`threshold_match_probability=0.05`.


In [ ]:
from pathlib import Path

import json

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import IntegerType
from splink.backends.spark import similarity_jar_location

DATA_DIR = Path('/data/spark/shared/datasets')
CENSO_PARQUET = DATA_DIR / 'censo_limpo_aplicacao.parquet'
CPF_PARQUET = DATA_DIR / 'cpf_limpo_aplicacao.parquet'
MODEL_JSON = DATA_DIR / 'splink_model_mae_endereco.json'
OUT_PARQUET = DATA_DIR / 'splink_predictions_mae_endereco_spark.parquet'

KEEP_COLS = [
    'unique_id',
    'nome_completo_phon',
    'primeiro_nome_phon',
    'ultimo_nome_phon',
    'data_nascimento',
    'ano_nascimento',
    'mes_nascimento',
    'dia_nascimento',
    'idade',
    'nome_mae_phon',
    'logradouro_norm',
    'cep',
    'cod_municipio',
    'sexo',
]

# JAR das UDFs (jaro_winkler, damerau_levenshtein, ...). Precisa no classpath.
JAR = similarity_jar_location()
spark = (
    SparkSession.builder
    .appName('02g_splink_mae_endereco_ma')
    .config('spark.jars', JAR)
    .getOrCreate()
)
spark.sparkContext.addJar(JAR)
print('similarity jar:', JAR)

print('DATA_DIR:', DATA_DIR)
print('modelo:', MODEL_JSON)
print('censo:', CENSO_PARQUET)
print('cpf:  ', CPF_PARQUET)
print('saida:', OUT_PARQUET)

import splink
print('spark.version:', spark.version)
print('splink:', getattr(splink, '__version__', '?'))


In [ ]:
df_censo = (
    spark.read.parquet(str(CENSO_PARQUET))
    .select(*KEEP_COLS)
    .withColumn('idade', F.col('idade').cast(IntegerType()))
    .withColumn('source_dataset', F.lit('censo'))
)
df_cpf = (
    spark.read.parquet(str(CPF_PARQUET))
    .select(*KEEP_COLS)
    .withColumn('idade', F.col('idade').cast(IntegerType()))
    .withColumn('source_dataset', F.lit('cpf'))
)

print('censo:', df_censo.count())
print('cpf:  ', df_cpf.count())
df_censo.select(
    'unique_id', 'nome_completo_phon', 'data_nascimento', 'idade'
).show(3, truncate=False)
df_cpf.select(
    'unique_id', 'nome_completo_phon', 'data_nascimento', 'idade'
).show(3, truncate=False)


In [ ]:
from splink import Linker, block_on
from splink.backends.spark import SparkAPI

blocking_rules = [
    block_on('nome_completo_phon'),
    block_on('primeiro_nome_phon', 'ultimo_nome_phon', 'ano_nascimento'),
    block_on('primeiro_nome_phon', 'ultimo_nome_phon', 'mes_nascimento', 'dia_nascimento'),
    block_on('primeiro_nome_phon', 'ultimo_nome_phon', 'mes_nascimento', 'ano_nascimento'),
    block_on('primeiro_nome_phon', 'data_nascimento'),
    block_on('ultimo_nome_phon', 'data_nascimento'),
    block_on('primeiro_nome_phon', 'mes_nascimento', 'dia_nascimento', 'cod_municipio'),
    block_on('primeiro_nome_phon', 'mes_nascimento', 'ano_nascimento', 'cod_municipio'),
    block_on('ultimo_nome_phon', 'mes_nascimento', 'dia_nascimento', 'sexo', 'cep'),
    block_on('ultimo_nome_phon', 'mes_nascimento', 'ano_nascimento', 'sexo', 'cep'),
    (
        "l.ultimo_nome_phon = r.ultimo_nome_phon "
        "AND l.ano_nascimento = r.ano_nascimento "
        "AND l.mes_nascimento = r.mes_nascimento "
        "AND l.sexo = r.sexo "
        "AND damerau_levenshtein(l.primeiro_nome_phon, r.primeiro_nome_phon) <= 2 "
        "AND least(len(l.primeiro_nome_phon), len(r.primeiro_nome_phon)) >= 6 "
        "AND damerau_levenshtein(l.primeiro_nome_phon, r.primeiro_nome_phon) * 6 "
        "<= least(len(l.primeiro_nome_phon), len(r.primeiro_nome_phon))"
    ),
    (
        "l.primeiro_nome_phon = r.primeiro_nome_phon "
        "AND l.ultimo_nome_phon = r.ultimo_nome_phon "
        "AND l.cod_municipio = r.cod_municipio "
        "AND l.sexo = r.sexo "
        "AND abs(l.idade - r.idade) <= 1"
    ),
    (
        "l.logradouro_norm = r.logradouro_norm "
        "AND l.cep = r.cep "
        "AND l.cod_municipio = r.cod_municipio "
        "AND l.sexo = r.sexo "
        "AND abs(l.idade - r.idade) <= 1"
    ),
    (
        "l.nome_mae_phon = r.nome_mae_phon "
        "AND l.cod_municipio = r.cod_municipio "
        "AND l.sexo = r.sexo "
        "AND abs(l.idade - r.idade) <= 1"
    ),
]

print('regras de blocking:', len(blocking_rules))


In [ ]:
import inspect

from splink import SettingsCreator

with open(MODEL_JSON) as f:
    settings = json.load(f)

settings['retain_intermediate_calculation_columns'] = False
settings['retain_matching_columns'] = False
settings['blocking_rules_to_generate_predictions'] = blocking_rules
settings.pop('sql_dialect', None)

# JSON salvo pode ter chaves que o SettingsCreator desta versao nao aceita
# (ex.: match_weight_column_prefix em alguns builds do Splink 5).
allowed = set(inspect.signature(SettingsCreator.__init__).parameters) - {'self'}
drop = sorted(k for k in settings if k not in allowed)
if drop:
    print('ignorando chaves do JSON:', drop)
settings = {k: v for k, v in settings.items() if k in allowed}

spark.sparkContext.setCheckpointDir(
    '/data/spark/shared/checkpoints'
)

# Materializar intermediarios em checkpoint,
# evitando o checkpoint RDD que estava falhando
db_api = SparkAPI(
    spark_session=spark,
    break_lineage_method='checkpoint',
    repartition_after_blocking=False,
)

linker = Linker(
    [df_censo, df_cpf],
    settings,
    db_api,
    input_table_aliases=['censo', 'cpf'],
)

print('Linker pronto:', MODEL_JSON)


In [ ]:
df_predict = linker.inference.predict(threshold_match_probability=0.05)
pred = df_predict.as_spark_dataframe()
print('predictions:', pred.count())
pred.select(
    'unique_id_l', 'unique_id_r', 'match_probability', 'match_weight'
).show(5)


In [ ]:
keep = [
    c for c in (
        'unique_id_l', 'unique_id_r', 'match_probability', 'match_weight'
    )
    if c in pred.columns
]

(
    pred.select(*keep)
    .write.mode('overwrite')
    .parquet(str(OUT_PARQUET))
)

print('Predictions:', OUT_PARQUET)
print('colunas:', keep)
